# 03 — ASR: Wav2Vec2 Fine-tuning (Local)

Fine-tunes `nguyenvulebinh/wav2vec2-base-vi-vlsp2020` trên ViMD.

**Target KPIs**: WER ≤ 8% | CER ≤ 4%


In [1]:
# ── Cell 0: Cấu hình đường dẫn local ──────────────────────────────────────────
import sys, os
from pathlib import Path

_nb_dir = Path(os.path.abspath('.')).resolve()
if _nb_dir.name == 'notebooks':
    PROJECT_ROOT = str(_nb_dir.parent.parent)
elif _nb_dir.name == 'training':
    PROJECT_ROOT = str(_nb_dir.parent)
else:
    PROJECT_ROOT = str(_nb_dir)
# PROJECT_ROOT = r'd:\FPTU\Semester 7\DAT301m\Project'

DATASET_PATH   = os.path.join(PROJECT_ROOT, 'storage', 'datasets', 'hf_dataset', 'vietspeech-vimd')
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, 'storage', 'checkpoints', 'asr_wav2vec2')
EXPORT_DIR     = os.path.join(PROJECT_ROOT, 'storage', 'weights', 'asr_wav2vec2')

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')


Device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [2]:
# ── Cell 1: Cài dependencies ────────────────────────────────────────────────────
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'datasets', 'accelerate', 'evaluate',
                'peft', 'jiwer', 'soundfile', 'mlflow'], check=False)
print('Dependencies installed')

Dependencies installed


In [3]:
# ── Cell 2: Load dataset ─────────────────────────────────────────────────────────
from datasets import load_from_disk

if not os.path.exists(DATASET_PATH):
    raise RuntimeError(f'Dataset không tìm thấy: {DATASET_PATH}\nHãy chạy 01_dataset_prep.ipynb trước.')

ds = load_from_disk(DATASET_PATH)
# ⚠️ FIX: KHÔNG cast_column audio (gây torchcodec bug + WSL deadlock)
# Audio sẽ được đọc on-the-fly trong collate_fn bằng BytesIO

# Xác định AUDIO_DIR — cùng thư mục mà 02_dialect_identification.ipynb đã giải nén vào
AUDIO_DIR = os.path.join(DATASET_PATH, 'audio')
if not os.path.exists(AUDIO_DIR) or len(os.listdir(AUDIO_DIR)) == 0:
    print(f'⚠️  AUDIO_DIR trống: {AUDIO_DIR}')
    print('Hãy chạy 02_dialect_identification.ipynb Cell 3 trước để giải nén audio.')
else:
    n = len(os.listdir(AUDIO_DIR))
    print(f'✓ AUDIO_DIR: {AUDIO_DIR} ({n} files)')

print('Dataset loaded:', ds)


/home/justeg4n/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ AUDIO_DIR: /mnt/d/FPTU/Semester 7/DAT301m/Project/storage/datasets/hf_dataset/vietspeech-vimd/audio (18083 files)
Dataset loaded: DatasetDict({
    train: Dataset({
        features: ['duration', 'mel_mean', 'mel_std', 'energy_mean', 'energy_std', 'energy_max', 'pitch_mean', 'pitch_std', 'pitch_min', 'pitch_max', 'voiced_ratio', 'zcr_mean', 'zcr_std', 'spectral_centroid_mean', 'speaker_embedding', 'filename', 'text', 'speakerID', 'gender', 'province', 'region', 'ethnicity', 'language', 'is_augmented', 'aug_id'],
        num_rows: 28814
    })
    valid: Dataset({
        features: ['duration', 'mel_mean', 'mel_std', 'energy_mean', 'energy_std', 'energy_max', 'pitch_mean', 'pitch_std', 'pitch_min', 'pitch_max', 'voiced_ratio', 'zcr_mean', 'zcr_std', 'spectral_centroid_mean', 'speaker_embedding', 'filename', 'text', 'speakerID', 'gender', 'province', 'region', 'ethnicity', 'language', 'is_augmented', 'aug_id'],
        num_rows: 3604
    })
    test: Dataset({
        features: ['durat

In [4]:
# ── Cell 3: Load Wav2Vec2 + Processor ─────────────────────────────────────────────
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

MODEL_NAME = 'nguyenvulebinh/wav2vec2-base-vi-vlsp2020'
print(f'Loading {MODEL_NAME}...')

processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)

# ⚠️ FIX: Không truyền vocab_size — model đã có vocab size đúng từ pretraining.
# Truyền vocab_size khác → mismatch lm_head → RuntimeError ignore_mismatched_sizes
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
)

if device == 'cuda':
    model = model.to(device)

# ⚡ FREEZE FEATURE EXTRACTOR (CNN layers) để tiết kiệm VRAM và chống overfit
model.freeze_feature_extractor()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'{MODEL_NAME} loaded')
print(f'Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')


Loading nguyenvulebinh/wav2vec2-base-vi-vlsp2020...


Loading weights: 100%|██████████| 213/213 [00:00<00:00, 10431.76it/s]
[transformers] Wav2Vec2ForCTC LOAD REPORT from: nguyenvulebinh/wav2vec2-base-vi-vlsp2020
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
feature_transform.linear2.bias            | UNEXPECTED |  | 
feature_transform.bn1.bias                | UNEXPECTED |  | 
feature_transform.bn3.bias                | UNEXPECTED |  | 
feature_transform.linear1.bias            | UNEXPECTED |  | 
feature_transform.bn2.num_batches_tracked | UNEXPECTED |  | 
feature_transform.linear1.weight          | UNEXPECTED |  | 
feature_transform.bn2.running_var         | UNEXPECTED |  | 
feature_transform.bn3.running_mean        | UNEXPECTED |  | 
feature_transform.linear3.bias            | UNEXPECTED |  | 
feature_transform.bn2.running_mean        | UNEXPECTED |  | 
feature_transform.linear2.weight          | UNEXPECTED |  | 
feature_transform.bn1.num_batches_tracked | UNEX

AttributeError: 'Wav2Vec2ForCTC' object has no attribute 'freeze_feature_extractor'

In [ ]:
# ── Cell 4: (Đã xóa LoRA, dùng Full Fine-tuning với Frozen Extractor) ────────

In [5]:
# ── Cell 5: Collate fn + DataLoader (On-the-fly, WSL deadlock-safe) ─────────────
import numpy as np
import librosa
import soundfile as sf
import io, os
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

MAX_DURATION_S = 8.0
MAX_SAMPLES    = int(MAX_DURATION_S * 16000)  # 128000 raw samples
MAX_LABEL_LEN  = 256

# ⚡ STRATEGY: Copy chỉ những file THỰC SỰ dùng trong dataset sang /tmp
# Audio dir = 56GB (toàn bộ) nhưng chỉ cần file có trong ds → tiết kiệm disk
# /tmp nằm trên WSL virtual disk (918GB trống) → an toàn
TMP_AUDIO_DIR = '/tmp/vimd_audio'
os.makedirs(TMP_AUDIO_DIR, exist_ok=True)

needed = set(ds['train']['filename']) | set(ds['valid']['filename']) | set(ds['test']['filename'])
already = set(os.listdir(TMP_AUDIO_DIR))
to_copy = [fn for fn in needed if fn not in already]

if to_copy:
    import shutil
    print(f'Copying {len(to_copy)} files to /tmp/vimd_audio (WSL disk 918GB free)...')
    for fn in tqdm(to_copy, desc='Copy to /tmp'):
        src_path = os.path.join(AUDIO_DIR, fn)
        if os.path.exists(src_path):
            shutil.copy2(src_path, os.path.join(TMP_AUDIO_DIR, fn))
    print(f'✓ Done. /tmp/vimd_audio: {len(os.listdir(TMP_AUDIO_DIR))} files')
else:
    print(f'✓ /tmp/vimd_audio đã đủ ({len(already)} files) — skip copy')

FAST_AUDIO_DIR = TMP_AUDIO_DIR

def collate_fn_wav2vec(batch):
    audio_arrays = []
    labels_list  = []

    for item in batch:
        fn   = item['filename']
        path = os.path.join(FAST_AUDIO_DIR, fn)

        # Fallback sang AUDIO_DIR nếu file chưa được copy
        if not os.path.exists(path):
            path = os.path.join(AUDIO_DIR, fn)

        with open(path, 'rb') as f:
            b = f.read()
        data, sr = sf.read(io.BytesIO(b), dtype='float32')

        if data.ndim > 1:
            data = data.mean(axis=1)
        if sr != 16000:
            data = librosa.resample(data, orig_sr=sr, target_sr=16000)
        data, _ = librosa.effects.trim(data, top_db=20)
        if len(data) == 0:
            data = np.zeros(1600, dtype=np.float32)
        data = data[:MAX_SAMPLES]
        audio_arrays.append(data)

        text_col = 'text' if 'text' in item else 'transcript'
        ids = processor.tokenizer(item[text_col]).input_ids[:MAX_LABEL_LEN]
        labels_list.append(ids)

    inputs = processor(
        audio_arrays,
        sampling_rate=16000,
        return_tensors='pt',
        padding=True,
    )
    max_label_len = max(len(l) for l in labels_list)
    padded_labels = [
        l + [-100] * (max_label_len - len(l))
        for l in labels_list
    ]
    inputs['labels'] = torch.tensor(padded_labels, dtype=torch.long)
    return inputs

# Chỉ dùng sample có file audio tồn tại
def has_audio(item):
    return os.path.exists(os.path.join(FAST_AUDIO_DIR, item['filename'])) or \
           os.path.exists(os.path.join(AUDIO_DIR, item['filename']))

ds_train = ds['train'].filter(has_audio, desc='Filter train')
ds_valid = ds['valid'].filter(has_audio, desc='Filter valid')
ds_test  = ds['test'].filter(has_audio,  desc='Filter test')
print(f'After filter — Train: {len(ds_train)} | Valid: {len(ds_valid)} | Test: {len(ds_test)}')

train_loader = DataLoader(ds_train, batch_size=8, shuffle=True,
    collate_fn=collate_fn_wav2vec, num_workers=0, pin_memory=False)
val_loader   = DataLoader(ds_valid, batch_size=8, shuffle=False,
    collate_fn=collate_fn_wav2vec, num_workers=0)
test_loader  = DataLoader(ds_test,  batch_size=8, shuffle=False,
    collate_fn=collate_fn_wav2vec, num_workers=0)

print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')


Copying 18083 files to /tmp/vimd_audio (WSL disk 918GB free)...


Copy to /tmp: 100%|██████████| 18083/18083 [21:12<00:00, 14.21it/s]  


✓ Done. /tmp/vimd_audio: 18083 files


Filter test: 100%|██████████| 3748/3748 [00:00<00:00, 5693.35 examples/s] 

After filter — Train: 28814 | Valid: 3604 | Test: 3748
Train batches: 3602 | Val: 451 | Test: 469


In [6]:
# ── Cell 6: Training Loop (CTC, manual loop) ────────────────────────────────────
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm
import mlflow, evaluate

wer_metric = evaluate.load('wer')

os.environ['MLFLOW_TRACKING_URI'] = os.path.join(PROJECT_ROOT, 'storage', 'logs', 'mlflow')
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'

NUM_EPOCHS    = 3
LR            = 1e-5
GRAD_ACC      = 8
MAX_GRAD_NORM = 1.0
USE_AMP       = False  # CTC + fp16 = NaN → tắt AMP

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=0.01
)
total_steps = NUM_EPOCHS * (len(train_loader) // GRAD_ACC)
scheduler   = CosineAnnealingLR(optimizer, T_max=max(total_steps, 1))
global_step = 0

mlflow.set_experiment('asr-wav2vec2')

with mlflow.start_run():
    mlflow.log_params({
        'model': MODEL_NAME, 'lr': LR, 'epochs': NUM_EPOCHS,
        'batch_size': train_loader.batch_size, 'grad_acc': GRAD_ACC,
    })
    best_wer = float('inf')
    nan_count = 0

    for epoch in range(NUM_EPOCHS):
        model.train()
        total_loss  = 0.0
        valid_steps = 0
        optimizer.zero_grad()

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}', dynamic_ncols=True)
        for step, batch in enumerate(pbar):
            input_values = batch['input_values'].to(device)
            labels       = batch['labels'].to(device)

            outputs = model(input_values=input_values, labels=labels)
            loss    = outputs.loss / GRAD_ACC

            if torch.isnan(loss) or torch.isinf(loss):
                nan_count += 1
                optimizer.zero_grad()
                pbar.set_postfix({'loss': 'NaN', 'skip': nan_count})
                continue

            loss.backward()
            total_loss  += loss.item() * GRAD_ACC
            valid_steps += 1

            if (step + 1) % GRAD_ACC == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

            # ⚡ Update tqdm every step (thay vì print every 100)
            pbar.set_postfix({
                'loss': f'{loss.item()*GRAD_ACC:.4f}',
                'lr':   f'{scheduler.get_last_lr()[0]:.1e}',
                'nan':  nan_count,
                'step': global_step,
            })

        avg_loss = total_loss / valid_steps if valid_steps > 0 else float('nan')

        # ── Validate ──
        model.eval()
        all_preds, all_refs = [], []
        with torch.no_grad():
            for vbatch in tqdm(val_loader, desc='Validating', leave=False):
                input_values = vbatch['input_values'].to(device)
                logits   = model(input_values=input_values).logits
                pred_ids = logits.argmax(dim=-1)
                preds    = processor.batch_decode(pred_ids)
                vlabels  = vbatch['labels'].clone()
                vlabels[vlabels == -100] = processor.tokenizer.pad_token_id
                refs = processor.batch_decode(vlabels, group_tokens=False)
                all_preds.extend(preds)
                all_refs.extend(refs)

        wer = 100 * wer_metric.compute(predictions=all_preds, references=all_refs)
        mlflow.log_metrics({'train_loss': avg_loss, 'val_wer': wer,
                            'nan_skipped': nan_count}, step=epoch)
        print(f'\nEpoch {epoch+1}/{NUM_EPOCHS}: loss={avg_loss:.4f} | val_WER={wer:.2f}% | NaN_skip={nan_count}')

        if wer < best_wer:
            best_wer = wer
            model.save_pretrained(CHECKPOINT_DIR)
            processor.save_pretrained(CHECKPOINT_DIR)
            print(f'  → Checkpoint saved (WER={wer:.2f}%)')

    print(f'\nTraining done. Best WER: {best_wer:.2f}% | NaN skipped: {nan_count}')
    mlflow.log_metric('best_val_wer', best_wer)


Epoch 1/3: 100%|██████████| 3602/3602 [9:07:44<00:00,  9.12s/it, loss=3.4561, lr=7.5e-06, nan=0, step=450]  



Epoch 1/3: loss=3.8910 | val_WER=100.00% | NaN_skip=0


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.49s/it]


  → Checkpoint saved (WER=100.00%)


Epoch 2/3: 100%|██████████| 3602/3602 [9:27:00<00:00,  9.44s/it, loss=3.4227, lr=2.5e-06, nan=0, step=900]  



Epoch 2/3: loss=3.3820 | val_WER=100.00% | NaN_skip=0


Epoch 3/3: 100%|██████████| 3602/3602 [9:20:41<00:00,  9.34s/it, loss=3.3225, lr=0.0e+00, nan=0, step=1350]   



Epoch 3/3: loss=3.3657 | val_WER=100.00% | NaN_skip=0

Training done. Best WER: 100.00% | NaN skipped: 0


In [7]:
# ── Cell 7: Save final model ────────────────────────────────────────────────────
import shutil

# Copy checkpoint tốt nhất vào EXPORT_DIR
if os.path.exists(CHECKPOINT_DIR):
    os.makedirs(EXPORT_DIR, exist_ok=True)
    for fn in os.listdir(CHECKPOINT_DIR):
        shutil.copy2(os.path.join(CHECKPOINT_DIR, fn), EXPORT_DIR)

print(f'✓ Wav2Vec2 ASR model saved to: {EXPORT_DIR}')
print(f'Files: {os.listdir(EXPORT_DIR)}')


✓ Wav2Vec2 ASR model saved to: /mnt/d/FPTU/Semester 7/DAT301m/Project/storage/weights/asr_wav2vec2
Files: ['added_tokens.json', 'config.json', 'model.safetensors', 'processor_config.json', 'tokenizer_config.json', 'vocab.json']


In [9]:
# ── Cell 8: Evaluation on test set ───────────────────────────────────────────────
model.eval()
all_preds, all_refs = [], []

with torch.no_grad():
    for batch in test_loader:
        input_values = batch['input_values'].to(device)
        logits = model(input_values=input_values).logits
        pred_ids = logits.argmax(dim=-1)
        preds = processor.batch_decode(pred_ids)
        labels = batch['labels']
        labels[labels == -100] = processor.tokenizer.pad_token_id
        refs = processor.batch_decode(labels, group_tokens=False)
        all_preds.extend(preds)
        all_refs.extend(refs)

test_wer = 100 * wer_metric.compute(predictions=all_preds, references=all_refs)
print(f'Test WER : {test_wer:.2f}%')
print(f'Target   : ≤ 8.0% → {"PASS ✅" if test_wer <= 8.0 else "FAIL ❌"}')


Test WER : 100.00%
Target   : ≤ 8.0% → FAIL ❌


## ✓ Wav2Vec2 ASR Fine-tuning Complete

**Model saved to**: `storage/weights/asr_wav2vec2/`

**Next**: Run `04_translation.ipynb`